In [1]:
import pandas as pd
import numpy as np
import gc
import io
import os
from itertools import combinations
from tqdm import tqdm

from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

pd.reset_option('display.float_format')
pd.set_option('display.max_colwidth', None)

from config import ROOT, prev_num_aggregations  # lib này được khởi tạo ban đầu dự án

import helpers.view as view
import helpers.EDA as EDA
import modules.utils as utils
import modules.encode as encode

from helpers.cache_clear import cache_clear

get_pickle = utils.get_pickle
get_pickles = utils.get_pickles

import lightgbm as lgb

print(lgb.__file__)

HEAD = 100000

SEED = 71

d:\Data Science\venv\Lib\site-packages\lightgbm\__init__.py


In [2]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import Pool, cpu_count

In [3]:
fea = ["f101", "f102", "f103", "f104", "f105", "f106", "f107", "f108"]

In [4]:
prefixes = ["f0"] + fea
fea = "f0_f1"
print(prefixes)
print(fea)

['f0', 'f101', 'f102', 'f103', 'f104', 'f105', 'f106', 'f107', 'f108']
f0_f101-2-3-4-5-6-7-8


In [11]:
import re
def read_feather_with_head(file_path):
    if os.path.getsize(file_path) > 100:
        return pd.read_feather(file_path).head(HEAD)
         
def sanitize_feature_name(name):
    return re.sub(r"[+(),. ]", "_", name)         
         
def read(filename):
    with open(filename, 'r') as f:
        features = [ROOT + "/data/feature/train/" + sanitize_feature_name(line).strip() + ".f" for line in f]
        return features

### loại bỏ các feature có phương sai quá thấp (<= 0.0001)

In [ ]:
def handle_low_variance(df, variance_threshold=0.0001):
    variances = df.var()
    to_drop = variances[variances <= variance_threshold].index.tolist()
    filtered_df = df.drop(to_drop, axis=1)
    return filtered_df, to_drop

feature_paths = utils.get_feature_paths(prefixes=prefixes)
chunk_size = 12
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

# multi thread
with ThreadPoolExecutor(max_workers=12) as executor, \
    open(os.path.join(ROOT, f".log/_used/high_var_{fea}.txt"), "w") as f_selected, \
    open(os.path.join(ROOT, f".log/low_variance/low_var_{fea}.txt"), "w") as f_unselected:
    
    selected_features_all = []
    unselected_features_all = []
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat(chunk_dfs, axis=1)

        X_filtered, unselected_features = handle_low_variance(X)
        selected_features = X_filtered.columns.tolist()
        
        selected_features_all.extend(selected_features)
        unselected_features_all.extend(unselected_features)
        
    f_selected.write("\n".join(selected_features_all)) # đọc ghi file nên chỉ làm 1 lần vì bất đồng bộ tốn thời gian
    f_unselected.write("\n".join(unselected_features_all))
    print(len(selected_features) + len(unselected_features))

### tìm các feature có tương quan thấp với target (<0.02). Sau đó tìm imp với tổ hợp các feature này vì có thể tổ hợp của chúng có thể mang lại tương quan target mạnh

In [12]:
def handle_low_correlation_with_target(df, target_column, threshold=0.02):
    correlations = df.corr()[target_column].abs().drop(target_column)
    low_corr_columns = correlations[correlations < threshold].index.tolist()
    return df.drop(low_corr_columns, axis=1), low_corr_columns

target = pd.read_feather(utils.get_TARGET_path()).head(HEAD)
target.columns = ["TARGET"]

feature_paths = read(os.path.join(ROOT, f".log/_used/high_var_{fea}.txt"))
chunk_size = 12
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

# multi thread: 20 phút => 1 phút 26s
with ThreadPoolExecutor(max_workers=12) as executor, \
    open(os.path.join(ROOT, f".log/_used/high_corr_target_{fea}.txt"), "w") as f_selected, \
    open(os.path.join(ROOT, f".log/_used/low_corr_target_{fea}.txt"), "w") as f_unselected:
    
    selected_features_all = []
    unselected_features_all = []
    i = 0
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat([pd.concat(chunk_dfs, axis=1), target], axis=1)

        X_filtered, unselected_features = handle_low_correlation_with_target(X, target_column="TARGET")
        selected_features = X_filtered.columns.tolist()
        selected_features.remove("TARGET")
        
        selected_features_all.extend(selected_features)
        unselected_features_all.extend(unselected_features)
        
        if i >= 1000:
            print(selected_features_all[-1])
            i=0
        i+=1
    f_selected.write("\n".join(selected_features_all)) # đọc ghi file nên chỉ làm 1 lần vì bất đồng bộ tốn thời gian
    f_unselected.write("\n".join(unselected_features_all))
    print(len(selected_features) + len(unselected_features))

f101_nyg_low_normal_NAME_CONTRACT_STATUS_Refused_mean
f102_nyg_high_total_debt_min
f103_approved_HOUR_APPR_PROCESS_START_mean
f103_refused_DAYS_FIRST_DUE_s_app_DAYS_LAST_PHONE_CHANGE_max_div_min
f104_nyg_low_normal_DAYS_DECISION_d_app_DAYS_LAST_PHONE_CHANGE_mean
1


### NOTE: ngoài ra rất nên  tìm các cặp feature tương quan cao với nhau, nhưng vì số lượng feature lớn nên trước hết tìm information gain của feature (dùng lightgbm) nhưng chạy 1p20s cho 600 feature rất lâu

In [14]:
def highly_correlated_pairs(df, threshold=0.98):
    corr_matrix = df.corr().abs().values  # tính một lần và chuyển thành numpy array
    cols = df.columns
    print("corr oke")
    high_corr_pairs = []
    correlated_features = set()

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if corr_matrix[i, j] >= threshold:
                high_corr_pairs.append((cols[i], cols[j]))
                correlated_features.add(cols[i])
                correlated_features.add(cols[j])
    uncorrelated_features = [col for col in cols if col not in correlated_features]

    return high_corr_pairs, uncorrelated_features

n_thread=12

feature_paths = read(os.path.join(ROOT, f".log/_used/high_var_{fea}.txt"))
chunk_size = n_thread * 50
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]
X=None
with ThreadPoolExecutor(max_workers=n_thread) as executor, \
    open(os.path.join(ROOT, f".log/_used/low_corr_{fea}.txt"), "w") as f_dummy, \
    open(os.path.join(ROOT, f".log/high_corr/high_corr_pair_{fea}.txt"), "w") as f_corr:

    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]

        X = pd.concat(chunk_dfs, axis=1)
        
        # high_corr_pairs, uncorrelated_features  = highly_correlated_pairs(X, threshold=0.98)
        se = []
        unse = []
        for col1, col2 in high_corr_pairs:
            unse.append(f"{col1},{col2}")
        for col in uncorrelated_features:
            se.append(f"{col}")
        f_corr.write("\n".join(unse)) # đọc ghi file nên chỉ làm 1 lần vì bất đồng bộ tốn thời gian
        f_dummy.write("\n".join(se))
        break
    print("done")

done
